# Saldo Disponible Real 2024 – 2026 · TATSA y TVL

Calcula el saldo disponible del presupuesto por **SKU + centro de costo + año**, con el mismo formato del Excel `Saldo_Disponible_Odoo_PQ`, y un informe de **RQ pendientes** (con OC y sin OC).

**Saldo = Presupuesto − (Compras + RQ_COC + RQ_SOC + Gasto_Sin_OC)**

| Columna | Qué es | De dónde sale |
|---|---|---|
| Presupuesto | Presupuesto vigente (ya incluye redistribuciones) | `crossovered.budget.lines.planned_amount` agrupado con `read_group` (trae todos los centros) |
| Compras | Lo ejecutado con OC: bienes = recibido neto de devoluciones; servicios = facturado | Líneas de OC confirmadas |
| RQ | Valor total pedido en OC confirmadas | Líneas de OC confirmadas |
| RQ_COC | En tránsito **con OC**: pedido − ejecutado | Líneas de OC abiertas |
| RQ_SOC | En tránsito **sin OC**: RQ aprobado − lo ya pedido en OC | Líneas de RQ abiertas |
| Gasto_Sin_OC | Facturas de proveedor sin OC (rendiciones, fondo fijo, gasto directo) | `account.move.line` sin `purchase_line_id` |
| Gasto_Rendicion | Parte del gasto sin OC que está dentro de una rendición | `account.surrender.requests.it` |

Reglas:
- **Año:** el del periodo de presupuesto del RQ (`periodo_id`); si la OC no tiene RQ, el año de aprobación de la OC. El gasto sin OC va por fecha contable. Así una importación facturada por el proveedor en otro año no se cuela en el año equivocado (caso G1010460).
- **Montos con IGV** según los impuestos reales de cada línea (como calcula Odoo), convertidos a soles con la tasa de la OC.
- **Unidades:** todo se lleva a la UdM del producto. Si el RQ, la OC o la recepción usan otra UdM se convierte y se anota en *Observaciones*.
- **Devoluciones a proveedor** se restan de lo recibido.
- Las columnas calculadas pesadas del presupuesto (`s_previsto`, `s_ordenado`, `planned_amount_computed`…) **no se usan**.
- Solo lectura: el notebook no escribe nada en Odoo.

##### Librerías

In [ ]:
import xmlrpc.client # Comunicar con Odoo
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from dotenv import load_dotenv
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

##### Parámetros

In [ ]:
ANIOS = [2024, 2025, 2026]                  # Años a calcular
EMPRESAS = {"TATSA": 2, "TVL": 1}           # Nombre -> id de compañía en Odoo
USUARIO = "JIM"                             # Usuario del .env (JIM, LUIS o ANTONIO). LUIS no puede leer presupuesto
FECHA_DESDE = "2023-07-01"                  # Ventana para traer RQ/OC/facturas que pueden caer en los años pedidos
TOLERANCIA = 0.0001                         # Cantidades menores se consideran cero
PRODUCTOS_EXCLUIDOS = {"ANTICIPO"}          # Líneas de factura que no son consumo de presupuesto

EXPORTAR_EXCEL = True                    # Cambiar a True para generar el Excel
ARCHIVO_EXCEL = "Saldo_Disponible_Real_2024_2026.xlsx"

##### Credenciales y conexión

In [ ]:
"""
    1. CONECTAR A ODOO - SE CREA OBJECTO PARA IDENTIFICAR + OBJECTO PARA HACER CONSULTAS
"""
load_dotenv() # Cargar las variables de entorno desde el archivo .env
url = os.getenv("URL_ODOO")
db = os.getenv("BD_ODOO")
username = os.getenv(f"USER_{USUARIO}")
password = os.getenv(f"PASSWORD_{USUARIO}")

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)


"""
    2. FUNCIONES DE LECTURA (SOLO LECTURA)
"""
def odoo(modelo, metodo, argumentos, opciones=None, empresas=None):
    # Equivalente a models.execute_kw con el contexto de idioma, zona horaria y compañías
    opciones = dict(opciones or {})
    contexto = {"lang": "es_PE", "tz": "America/Lima", **opciones.pop("context", {})}
    if empresas:
        contexto["allowed_company_ids"] = empresas
    opciones["context"] = contexto
    return models.execute_kw(db, uid, password, modelo, metodo, argumentos, opciones)


def leer_todo(modelo, dominio, campos, empresas, lote=5000, contexto=None):
    # search_read por bloques para no saturar la respuesta
    filas, desplazamiento = [], 0
    while True:
        bloque = odoo(modelo, "search_read", [dominio],
                      {"fields": campos, "limit": lote, "offset": desplazamiento, "order": "id", "context": contexto or {}},
                      empresas)
        filas += bloque
        if len(bloque) < lote:
            return filas
        desplazamiento += lote


def leer_ids(modelo, ids, campos, empresas, lote=2000):
    # read por bloques de ids
    ids, filas = list(ids), []
    for i in range(0, len(ids), lote):
        filas += odoo(modelo, "read", [ids[i:i + lote]], {"fields": campos}, empresas)
    return filas


def m2o_id(valor):
    return valor[0] if isinstance(valor, list) and valor else None


def m2o_nombre(valor):
    return valor[1] if isinstance(valor, list) and len(valor) == 2 else None

##### Tablas maestras: llaves, productos, unidades y años fiscales

In [ ]:
"""
    1. LLAVES DE GRUPO DE SKU
"""
df_llaves = pd.read_excel("Bd_Llaves_Actualizado.xlsx", engine="calamine")
df_llaves = df_llaves[["Cod_Producto", "Cod_Grupo_Sku"]].dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto")


"""
    2. PRODUCTOS (product.product, incluye archivados)
"""
productos = leer_todo(
    "product.product", [],
    ["default_code", "name", "numero_parte", "product_brand_id", "categ_id", "detailed_type", "uom_id"],
    [1, 2], contexto={"active_test": False}
)
df_productos = pd.DataFrame([{
    "product_id": p["id"],
    "Cod_Producto": p["default_code"] or None,
    "Producto": p["name"],
    "Nro_Parte": p.get("numero_parte") or None,
    "Marca": m2o_nombre(p.get("product_brand_id")),
    "Familia": (m2o_nombre(p["categ_id"]) or "").split("/")[0].strip() or None,
    "Tipo_Producto": p["detailed_type"],
    "udm_producto": m2o_id(p["uom_id"]),
} for p in productos])
df_productos = df_productos.merge(df_llaves, on="Cod_Producto", how="left")
df_productos["Cod_Grupo_Sku"] = df_productos["Cod_Grupo_Sku"].fillna(df_productos["Cod_Producto"])
PRODUCTO = df_productos.set_index("product_id").to_dict("index")


"""
    3. UNIDADES DE MEDIDA (para convertir todo a la UdM del producto)
"""
UDM = {u["id"]: {"nombre": u["name"], "factor": u["factor"] or 1.0, "categoria": m2o_id(u["category_id"])}
       for u in odoo("uom.uom", "search_read", [[]], {"fields": ["name", "factor", "category_id"], "context": {"active_test": False}})}


"""
    4. AÑOS FISCALES (id -> año)
"""
ANIO_FISCAL = {a["id"]: int(a["name"]) for a in odoo("account.fiscal.year", "search_read", [[]], {"fields": ["name"]})
               if str(a["name"]).isdigit()}


"""
    5. ETIQUETAS DE ESTADOS (valor técnico -> lo que se ve en Odoo)
"""
ESTADO_OC = dict(odoo("purchase.order", "fields_get", [["state_new"]], {"attributes": ["selection"]})["state_new"]["selection"])
ESTADO_RQ = dict(odoo("purchase.requisition", "fields_get", [["state"]], {"attributes": ["selection"]})["state"]["selection"])

print(f"Productos: {len(df_productos):,} | UdM: {len(UDM)} | Años fiscales: {sorted(ANIO_FISCAL.values())}")

##### Funciones de conversión

In [ ]:
def a_udm_producto(cantidad, udm_origen, product_id):
    """
        Convierte una cantidad a la UdM del producto (misma regla que Odoo: cantidad / factor_origen * factor_destino).
        Devuelve (cantidad_convertida, observación).
    """
    udm_destino = PRODUCTO.get(product_id, {}).get("udm_producto")
    if not udm_origen or not udm_destino or udm_origen == udm_destino:
        return cantidad, ""
    origen, destino = UDM.get(udm_origen), UDM.get(udm_destino)
    if origen is None or destino is None:
        return cantidad, "UdM desconocida"
    if origen["categoria"] != destino["categoria"]:
        return cantidad, f"UdM de otra categoría ({origen['nombre']} vs {destino['nombre']}), no se pudo convertir"
    return cantidad / origen["factor"] * destino["factor"], f"Convertido de {origen['nombre']} a {destino['nombre']}"


def unir_observaciones(serie):
    # Junta observaciones únicas en un texto
    vistas = []
    for texto in serie.dropna():
        for parte in str(texto).split(" | "):
            if parte and parte not in vistas:
                vistas.append(parte)
    return " | ".join(vistas)

##### 1. Presupuesto vigente

Se agrupa con `read_group` porque la búsqueda normal de líneas de presupuesto está modificada (módulo `presupuesto_view_details`) y esconde los centros administrativos. `planned_amount` ya trae las redistribuciones.

In [ ]:
def leer_presupuesto(empresa_id, anio):
    """
        1. AGRUPAR LÍNEAS DE PRESUPUESTO DEL AÑO POR SKU + CENTRO + SITUACIÓN
    """
    dominio = [
        ["company_id", "=", empresa_id],
        ["crossovered_budget_id.periodo_id.fiscal_year_id.name", "=", str(anio)],
        ["crossovered_budget_id.state", "not in", ["cancel", "draft"]],
    ]
    grupos = odoo("crossovered.budget.lines", "read_group",
                  [dominio, ["planned_amount"], ["product_id", "analytic_account_id", "general_budget_id"]],
                  {"lazy": False}, [empresa_id])
    df = pd.DataFrame([{
        "product_id": m2o_id(g["product_id"]),
        "centro_id": m2o_id(g["analytic_account_id"]),
        "Cuenta Analítica": m2o_nombre(g["analytic_account_id"]),
        "Situación presupuestaria": m2o_nombre(g["general_budget_id"]),
        "Presupuesto": g["planned_amount"] or 0.0,
    } for g in grupos])
    if df.empty:
        return df

    """
        2. UNA FILA POR SKU + CENTRO (LAS SITUACIONES SE JUNTAN EN TEXTO)
    """
    df = df.groupby(["product_id", "centro_id"], dropna=False).agg(
        **{"Cuenta Analítica": ("Cuenta Analítica", "first"),
           "Situación presupuestaria": ("Situación presupuestaria", lambda s: ", ".join(sorted(set(s.dropna())))),
           "Presupuesto": ("Presupuesto", "sum")}
    ).reset_index()
    df["Periodo"] = anio
    return df


presupuesto = {}
for nombre, empresa_id in EMPRESAS.items():
    presupuesto[nombre] = pd.concat([leer_presupuesto(empresa_id, anio) for anio in ANIOS], ignore_index=True)
    print(nombre, presupuesto[nombre].groupby("Periodo")["Presupuesto"].sum().round(0).to_dict())

##### 2. Requerimientos y sus líneas

In [ ]:
def leer_requerimientos(empresa_id):
    """
        1. CABECERAS DE RQ (AÑO = PERIODO DE PRESUPUESTO DEL RQ; SI NO TIENE, AÑO DE CREACIÓN)
    """
    cabeceras = leer_todo(
        "purchase.requisition",
        [["company_id", "=", empresa_id], ["create_date", ">=", FECHA_DESDE]],
        ["name", "state", "periodo_id", "tipo_requerimiento", "importacion_check", "create_date",
         "centro_costo", "requisition_origen_company_id", "state_calculatted_saved"],
        [empresa_id])
    df_rq = pd.DataFrame([{
        "requisition_id": r["id"],
        "Cod_RQ": r["name"],
        "estado_rq": r["state"],
        "Estado_RQ": ESTADO_RQ.get(r["state"], r["state"]),
        "Tipo_RQ": r["tipo_requerimiento"],
        "Fecha_RQ": r["create_date"][:10],
        "Anio": ANIO_FISCAL.get(m2o_id(r["periodo_id"])) or int(r["create_date"][:4]),
        "centro_rq": m2o_id(r["centro_costo"]),
        "rq_importacion": r["importacion_check"],
        "rq_hijo_de": m2o_nombre(r["requisition_origen_company_id"]),
    } for r in cabeceras])

    """
        2. LÍNEAS DE RQ (CANTIDAD EN UdM DEL PRODUCTO)
    """
    lineas = leer_todo(
        "purchase.requisition.line",
        [["company_id", "=", empresa_id], ["requisition_id.create_date", ">=", FECHA_DESDE], ["product_id", "!=", False]],
        ["requisition_id", "product_id", "product_uom_id", "product_qty", "price_unit", "price_total", "account_analytic_id"],
        [empresa_id])
    filas = []
    for l in lineas:
        product_id = m2o_id(l["product_id"])
        cantidad, obs_udm = a_udm_producto(l["product_qty"] or 0.0, m2o_id(l["product_uom_id"]), product_id)
        filas.append({
            "rq_line_id": l["id"],
            "requisition_id": m2o_id(l["requisition_id"]),
            "product_id": product_id,
            "centro_linea_rq": m2o_id(l["account_analytic_id"]),
            "udm_rq": m2o_nombre(l["product_uom_id"]),
            "Cant_RQ": cantidad,
            "precio_rq_unit": l["price_unit"] or 0.0,
            "Valor_RQ_Soles": l["price_total"] or 0.0,   # "Precio en Soles" del RQ, con impuestos
            "obs_udm_rq": obs_udm,
        })
    df_lineas = pd.DataFrame(filas).merge(df_rq, on="requisition_id", how="left")
    return df_rq, df_lineas


requerimientos = {nombre: leer_requerimientos(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, (df_rq, df_rql) in requerimientos.items():
    print(f"{nombre}: {len(df_rq):,} RQ | {len(df_rql):,} líneas")

##### 3. Órdenes de compra, devoluciones y facturas de OC

In [ ]:
def leer_compras(empresa_id):
    """
        1. CABECERAS DE OC (SIN CANCELADAS)
    """
    ordenes = leer_todo(
        "purchase.order",
        [["company_id", "=", empresa_id], ["date_order", ">=", FECHA_DESDE], ["state", "!=", "cancel"]],
        ["name", "state", "state_new", "currency_id", "currency_rate", "date_approve", "date_order",
         "partner_id", "requisition_id", "origin", "importacion_check"],
        [empresa_id])
    df_oc = pd.DataFrame([{
        "order_id": o["id"],
        "Cod_OC": o["name"],
        "estado_oc": o["state"],
        "estado_oc_new": o["state_new"],
        "Estado_OC": ESTADO_OC.get(o["state_new"], o["state_new"]),
        "Moneda": m2o_nombre(o["currency_id"]),
        "tasa": o["currency_rate"] if o["currency_rate"] else 1.0,
        "Fecha_OC": (o["date_approve"] or o["date_order"])[:10],
        "Proveedor": m2o_nombre(o["partner_id"]),
        "oc_requisition_id": m2o_id(o["requisition_id"]),
        "oc_importacion": o["importacion_check"],
    } for o in ordenes])

    """
        2. LÍNEAS DE OC
    """
    lineas = leer_todo(
        "purchase.order.line",
        [["company_id", "=", empresa_id], ["order_id.date_order", ">=", FECHA_DESDE], ["state", "!=", "cancel"], ["product_id", "!=", False]],
        ["order_id", "product_id", "product_uom", "product_qty", "qty_received", "qty_invoiced",
         "price_total", "account_analytic_id", "requisition_line_newid", "qty_received_method"],
        [empresa_id])
    df_ocl = pd.DataFrame([{
        "po_line_id": l["id"],
        "order_id": m2o_id(l["order_id"]),
        "product_id": m2o_id(l["product_id"]),
        "udm_oc_id": m2o_id(l["product_uom"]),
        "udm_oc": m2o_nombre(l["product_uom"]),
        "cant_oc_linea": l["product_qty"] or 0.0,
        "recibido_linea": l["qty_received"] or 0.0,
        "facturado_linea": l["qty_invoiced"] or 0.0,
        "total_linea": l["price_total"] or 0.0,          # en moneda de la OC, con impuestos
        "centro_oc": m2o_id(l["account_analytic_id"]),
        "rq_line_id": m2o_id(l["requisition_line_newid"]),
        "metodo_recepcion": l["qty_received_method"],
    } for l in lineas]).merge(df_oc, on="order_id", how="left")

    """
        3. DEVOLUCIONES A PROVEEDOR (MOVIMIENTOS HECHOS HACIA UBICACIÓN DE PROVEEDOR)
           to_refund = True -> Odoo ya las restó de lo recibido; False -> se restan aquí
    """
    devoluciones = leer_todo(
        "stock.move",
        [["company_id", "=", empresa_id], ["state", "=", "done"], ["purchase_line_id", "!=", False],
         ["location_dest_id.usage", "=", "supplier"], ["date", ">=", FECHA_DESDE]],
        ["purchase_line_id", "product_uom_qty", "product_uom", "to_refund", "picking_id"],
        [empresa_id])
    df_dev = pd.DataFrame([{
        "po_line_id": m2o_id(d["purchase_line_id"]),
        "cant_dev_linea": d["product_uom_qty"] or 0.0,
        "udm_dev": m2o_id(d["product_uom"]),
        "dev_restada_por_odoo": d["to_refund"],
        "albaran_dev": m2o_nombre(d["picking_id"]),
    } for d in devoluciones])

    """
        4. FACTURAS DE PROVEEDOR LIGADAS A OC (MONTO EN SOLES CON IMPUESTOS Y FECHA DE EMISIÓN)
    """
    apuntes = leer_todo(
        "account.move.line",
        [["company_id", "=", empresa_id], ["purchase_line_id", "!=", False], ["parent_state", "=", "posted"],
         ["exclude_from_invoice_tab", "=", False], ["date", ">=", FECHA_DESDE]],
        ["purchase_line_id", "move_id", "balance", "price_subtotal", "price_total"],
        [empresa_id])
    asientos = {a["id"]: a for a in leer_ids("account.move", {m2o_id(a["move_id"]) for a in apuntes},
                                             ["name", "ref", "invoice_date", "move_type"], [empresa_id])}
    filas = []
    for a in apuntes:
        asiento = asientos.get(m2o_id(a["move_id"]), {})
        if asiento.get("move_type") not in ("in_invoice", "in_refund", "in_receipt"):
            continue
        ratio = (a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0
        filas.append({
            "po_line_id": m2o_id(a["purchase_line_id"]),
            "facturado_soles": (a["balance"] or 0.0) * ratio,
            "Factura": asiento.get("name"),
            "fecha_emision": asiento.get("invoice_date"),
        })
    df_fact = pd.DataFrame(filas, columns=["po_line_id", "facturado_soles", "Factura", "fecha_emision"])
    return df_ocl, df_dev, df_fact


compras = {nombre: leer_compras(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, (df_ocl, df_dev, df_fact) in compras.items():
    print(f"{nombre}: {df_ocl['order_id'].nunique():,} OC | {len(df_ocl):,} líneas | {len(df_dev):,} devoluciones | {len(df_fact):,} líneas de factura")

##### 4. Gasto sin OC (facturas directas, rendiciones y fondo fijo)

In [ ]:
def leer_gasto_sin_oc(empresa_id):
    """
        1. MOVIMIENTOS QUE ESTÁN DENTRO DE UNA RENDICIÓN
    """
    rendiciones = leer_todo("account.surrender.requests.it", [["company_id", "=", empresa_id]], ["name", "move_ids"], [empresa_id])
    asiento_rendicion = {m: r["name"] for r in rendiciones for m in r["move_ids"]}

    """
        2. LÍNEAS DE FACTURA DE PROVEEDOR SIN OC, CON PRODUCTO (POR FECHA CONTABLE)
    """
    apuntes = leer_todo(
        "account.move.line",
        [["company_id", "=", empresa_id], ["parent_state", "=", "posted"], ["purchase_line_id", "=", False],
         ["product_id", "!=", False], ["exclude_from_invoice_tab", "=", False],
         ["move_id.move_type", "in", ["in_invoice", "in_refund", "in_receipt"]],
         ["date", ">=", f"{min(ANIOS)}-01-01"], ["date", "<=", f"{max(ANIOS)}-12-31"]],
        ["product_id", "analytic_account_id", "balance", "price_subtotal", "price_total", "date", "move_id"],
        [empresa_id])
    filas = []
    for a in apuntes:
        if PRODUCTO.get(m2o_id(a["product_id"]), {}).get("Cod_Producto") in PRODUCTOS_EXCLUIDOS:
            continue
        ratio = (a["price_total"] / a["price_subtotal"]) if a["price_subtotal"] else 1.0
        rendicion = asiento_rendicion.get(m2o_id(a["move_id"]))
        filas.append({
            "Periodo": int(a["date"][:4]),
            "product_id": m2o_id(a["product_id"]),
            "centro_id": m2o_id(a["analytic_account_id"]),
            "Cuenta Analítica": m2o_nombre(a["analytic_account_id"]),
            "Gasto_Sin_OC": (a["balance"] or 0.0) * ratio,
            "Gasto_Rendicion": (a["balance"] or 0.0) * ratio if rendicion else 0.0,
            "obs": "Gasto sin RQ/OC por rendición" if rendicion else "Gasto sin RQ/OC por factura directa",
        })
    return pd.DataFrame(filas)


gasto_sin_oc = {nombre: leer_gasto_sin_oc(empresa_id) for nombre, empresa_id in EMPRESAS.items()}
for nombre, df in gasto_sin_oc.items():
    print(nombre, df.groupby("Periodo")[["Gasto_Sin_OC", "Gasto_Rendicion"]].sum().round(0).to_dict("index"))

##### 5. Cálculo por línea de OC (ejecutado y en tránsito con OC)

- **Bien:** ejecutado = recibido neto de devoluciones. **Servicio** (o recepción manual): ejecutado = facturado, porque nunca se recepciona.
- **Pendiente con OC:** pedido − ejecutado, solo si la OC sigue abierta (`Orden de Compra`, `Bloqueado`, `Cerrado Parcialmente`). Si está `Cerrado` o `Cerrada No Completada` el pendiente es 0 y se anota.

In [ ]:
def calcular_oc(nombre):
    df_rq, df_rql = requerimientos[nombre]
    df_ocl, df_dev, df_fact = compras[nombre]
    df = df_ocl.copy()

    """
        1. RQ DE CADA LÍNEA (POR LÍNEA DE RQ O, SI NO, POR EL RQ DE LA OC + PRODUCTO)
    """
    rq_por_linea = df_rql.set_index("rq_line_id")[["requisition_id", "centro_linea_rq", "udm_rq", "obs_udm_rq"]].to_dict("index")
    rq_por_cabecera = df_rq.set_index("requisition_id").to_dict("index")
    df["requisition_id"] = [
        rq_por_linea.get(r, {}).get("requisition_id") if pd.notna(r) else o
        for r, o in zip(df["rq_line_id"], df["oc_requisition_id"])
    ]
    df["requisition_id"] = df["requisition_id"].fillna(df["oc_requisition_id"])
    for campo in ["Cod_RQ", "Estado_RQ", "Tipo_RQ", "Anio", "centro_rq", "rq_hijo_de"]:
        df[campo] = df["requisition_id"].map(lambda r: rq_por_cabecera.get(r, {}).get(campo) if pd.notna(r) else None)
    df["udm_rq"] = df["rq_line_id"].map(lambda r: rq_por_linea.get(r, {}).get("udm_rq") if pd.notna(r) else None)

    """
        2. AÑO Y CENTRO DE COSTO
    """
    df["Periodo"] = df["Anio"].fillna(df["Fecha_OC"].str[:4].astype(int)).astype(int)
    centro_linea_rq = df["rq_line_id"].map(lambda r: rq_por_linea.get(r, {}).get("centro_linea_rq") if pd.notna(r) else None)
    df["centro_id"] = df["centro_oc"].fillna(centro_linea_rq).fillna(df["centro_rq"])

    """
        3. CANTIDADES EN UdM DEL PRODUCTO
    """
    conv = [a_udm_producto(1.0, u, p) for u, p in zip(df["udm_oc_id"], df["product_id"])]
    df["factor_udm"] = [c[0] for c in conv]
    df["obs_udm"] = [c[1] for c in conv]

    # Devoluciones que Odoo no restó (to_refund = False)
    if not df_dev.empty:
        df_dev = df_dev.copy()
        no_restadas = df_dev[~df_dev["dev_restada_por_odoo"]].groupby("po_line_id")["cant_dev_linea"].sum()
        todas = df_dev.groupby("po_line_id").agg(cant_dev=("cant_dev_linea", "sum"),
                                                 albaranes_dev=("albaran_dev", lambda s: ", ".join(sorted(set(s.dropna())))))
        df["dev_no_restada"] = df["po_line_id"].map(no_restadas).fillna(0.0)
        df = df.merge(todas, left_on="po_line_id", right_index=True, how="left")
    else:
        df["dev_no_restada"], df["cant_dev"], df["albaranes_dev"] = 0.0, np.nan, None

    df["es_servicio"] = (df["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Tipo_Producto")) == "service") | (df["metodo_recepcion"] == "manual")
    df["Cant_OC"] = df["cant_oc_linea"] * df["factor_udm"]
    df["Cant_Recibida"] = (df["recibido_linea"] - df["dev_no_restada"]) * df["factor_udm"]
    df["Cant_Facturada"] = df["facturado_linea"] * df["factor_udm"]
    df["Cant_Ejecutada"] = np.where(df["es_servicio"], df["Cant_Facturada"], df["Cant_Recibida"])

    """
        4. PRECIO UNITARIO EN SOLES CON IMPUESTOS (TOTAL DE LÍNEA / CANTIDAD / TASA DE LA OC)
    """
    df["precio_soles"] = np.where(df["Cant_OC"] > 0, df["total_linea"] / df["tasa"] / df["Cant_OC"].where(df["Cant_OC"] > 0, 1), 0.0)

    """
        5. FACTURAS: MONTO EN SOLES Y FECHA DE EMISIÓN
    """
    if not df_fact.empty:
        fact = df_fact.groupby("po_line_id").agg(
            facturado_soles=("facturado_soles", "sum"),
            Facturas=("Factura", lambda s: ", ".join(sorted(set(s.dropna())))),
            fechas_emision=("fecha_emision", lambda s: sorted(set(x for x in s if x))))
        df = df.merge(fact, left_on="po_line_id", right_index=True, how="left")
    else:
        df["facturado_soles"], df["Facturas"], df["fechas_emision"] = np.nan, None, None
    df["facturado_soles"] = df["facturado_soles"].fillna(0.0)

    """
        6. EJECUTADO, PEDIDO Y PENDIENTE EN SOLES
    """
    confirmada = df["estado_oc"].isin(["purchase", "done"])
    abierta = confirmada & df["estado_oc_new"].isin(["purchase", "done", "parcialmente_cerrado"])
    df["Valor_OC"] = np.where(confirmada, df["Cant_OC"] * df["precio_soles"], 0.0)
    df["Valor_Ejecutado"] = np.where(
        confirmada,
        np.where(df["es_servicio"] & (df["facturado_soles"] != 0), df["facturado_soles"], df["Cant_Ejecutada"] * df["precio_soles"]),
        0.0)
    df["Cant_Pendiente"] = np.where(abierta, (df["Cant_OC"] - df["Cant_Ejecutada"]).clip(lower=0), 0.0)
    df["Cant_Pendiente"] = np.where(df["Cant_Pendiente"] < TOLERANCIA, 0.0, df["Cant_Pendiente"])
    df["Valor_Pendiente"] = df["Cant_Pendiente"] * df["precio_soles"]
    df["confirmada"] = confirmada

    """
        7. OBSERVACIONES
    """
    def observar(r):
        obs = []
        if r["Moneda"] and r["Moneda"] != "PEN":
            tc = 1 / r["tasa"] if r["tasa"] else 0
            obs.append(f"OC en {r['Moneda']} (TC {tc:.3f})")
        if r["oc_importacion"]:
            obs.append("Importación")
        if r["obs_udm"]:
            obs.append(f"UdM OC: {r['obs_udm']}")
        if isinstance(r["udm_rq"], str) and isinstance(r["udm_oc"], str) and r["udm_rq"] != r["udm_oc"]:
            obs.append(f"Pedido en RQ con {r['udm_rq']} y en OC con {r['udm_oc']}")
        if pd.notna(r["cant_dev"]) and r["cant_dev"]:
            obs.append(f"Devolución a proveedor {r['cant_dev'] * r['factor_udm']:g} ({r['albaranes_dev']})")
        if r["es_servicio"]:
            obs.append("Servicio: avance por facturación")
        if r["Cant_Ejecutada"] > r["Cant_OC"] + TOLERANCIA:
            obs.append("Ejecutado mayor a lo pedido")
        if r["confirmada"] and r["estado_oc_new"] == "cerrado_no" and r["Cant_OC"] - r["Cant_Ejecutada"] > TOLERANCIA:
            obs.append(f"OC cerrada no completada: {r['Cant_OC'] - r['Cant_Ejecutada']:g} sin ejecutar que no se cuentan")
        if r["Cant_Pendiente"] > 0 and (pd.Timestamp.today() - pd.Timestamp(r["Fecha_OC"])).days > 365:
            obs.append("OC abierta con más de 12 meses: revisar si se debe cerrar")
        if r["confirmada"] and r["estado_oc_new"] == "cerrado" and r["Cant_OC"] - r["Cant_Ejecutada"] > TOLERANCIA:
            obs.append(f"OC cerrada con {r['Cant_OC'] - r['Cant_Ejecutada']:g} sin ejecutar")
        if isinstance(r["fechas_emision"], list):
            otros = [f for f in r["fechas_emision"] if int(f[:4]) != r["Periodo"]]
            if otros:
                obs.append(f"Factura emitida el {', '.join(otros)}: Odoo la imputa a {', '.join(sorted({f[:4] for f in otros}))}, aquí va a {r['Periodo']}")
        if r["rq_hijo_de"]:
            obs.append(f"RQ relacionado de {r['rq_hijo_de']}")
        if r["Proveedor"] and r["Proveedor"].startswith(("TATSA PERU", "TRANSVIAL LIMA")):
            obs.append("OC a empresa relacionada")
        if pd.isna(r["requisition_id"]):
            obs.append("OC sin RQ")
        return " | ".join(obs)

    df["Observaciones"] = df.apply(observar, axis=1)
    return df


calculo_oc = {nombre: calcular_oc(nombre) for nombre in EMPRESAS}
for nombre, df in calculo_oc.items():
    c = df[df["confirmada"]]
    print(f"{nombre}: OC confirmadas {c['order_id'].nunique():,} | pendientes con OC {(c['Cant_Pendiente'] > 0).sum():,} líneas")

##### 6. RQ sin OC (en tránsito sin OC)

Líneas de RQ aprobadas (`Confirmado`, `En proceso`, `Selección del licitador`) menos lo ya pedido en OC confirmadas. Si el RQ trae precio referencial (≤ 1, típico cuando se compra en dólares) se usa el último precio de compra en soles.

In [ ]:
def calcular_rq_sin_oc(nombre):
    df_rq, df_rql = requerimientos[nombre]
    df_oc = calculo_oc[nombre]

    """
        1. LO YA PEDIDO EN OC CONFIRMADAS POR LÍNEA DE RQ (O POR RQ + PRODUCTO SI LA OC NO GUARDÓ LA LÍNEA)
    """
    confirmadas = df_oc[df_oc["confirmada"]]
    pedido_linea = confirmadas.dropna(subset=["rq_line_id"]).groupby("rq_line_id")["Cant_OC"].sum()
    pedido_rq_prod = confirmadas[confirmadas["rq_line_id"].isna()].groupby(["requisition_id", "product_id"])["Cant_OC"].sum()
    cotizaciones = df_oc[~df_oc["confirmada"]].groupby("requisition_id")["Cod_OC"].agg(lambda s: ", ".join(sorted(set(s))))

    """
        2. ÚLTIMO PRECIO DE COMPRA EN SOLES POR PRODUCTO (PARA RQ CON PRECIO REFERENCIAL)
    """
    ultimo_precio = (confirmadas[confirmadas["precio_soles"] > 0].sort_values("Fecha_OC")
                     .groupby("product_id")["precio_soles"].last())

    df = df_rql[df_rql["estado_rq"].isin(["in_progress", "ongoing", "open"]) & df_rql["Anio"].isin(ANIOS)].copy()
    df["Cant_Pedida_OC"] = df["rq_line_id"].map(pedido_linea).fillna(0.0)
    df["Cant_Pedida_OC"] += [pedido_rq_prod.get((r, p), 0.0) for r, p in zip(df["requisition_id"], df["product_id"])]
    df["Cant_Pendiente"] = (df["Cant_RQ"] - df["Cant_Pedida_OC"]).clip(lower=0)
    df = df[df["Cant_Pendiente"] > TOLERANCIA].copy()

    """
        3. VALOR EN SOLES CON IMPUESTOS
    """
    precio_rq = np.where(df["Cant_RQ"] > 0, df["Valor_RQ_Soles"] / df["Cant_RQ"].where(df["Cant_RQ"] > 0, 1), 0.0)
    referencial = (df["precio_rq_unit"] <= 1) | (precio_rq <= 0)
    df["precio_soles"] = np.where(referencial, df["product_id"].map(ultimo_precio).fillna(0.0), precio_rq)
    df["Valor_Pendiente"] = df["Cant_Pendiente"] * df["precio_soles"]

    def observar(r, ref):
        obs = []
        if r["obs_udm_rq"]:
            obs.append(f"UdM RQ: {r['obs_udm_rq']}")
        if ref:
            obs.append("Precio del RQ referencial: se usó el último precio de compra" if r["precio_soles"] > 0 else "Precio del RQ referencial y sin compras previas: valor 0")
        if r["Cant_Pedida_OC"] > 0:
            obs.append(f"Ya pedido en OC {r['Cant_Pedida_OC']:g} de {r['Cant_RQ']:g}")
        cot = cotizaciones.get(r["requisition_id"])
        if cot:
            obs.append(f"Cotización en curso: {cot}")
        if r["rq_importacion"]:
            obs.append("Importación")
        if r["Tipo_RQ"] == "servicio":
            obs.append("Servicio: avance por facturación")
        if r["rq_hijo_de"]:
            obs.append(f"RQ relacionado de {r['rq_hijo_de']}")
        return " | ".join(obs)

    df["Observaciones"] = [observar(r, ref) for (_, r), ref in zip(df.iterrows(), referencial)]
    df["centro_id"] = df["centro_linea_rq"].fillna(df["centro_rq"])
    df["Periodo"] = df["Anio"].astype(int)
    df["es_servicio"] = (df["Tipo_RQ"] == "servicio") | (df["product_id"].map(lambda p: PRODUCTO.get(p, {}).get("Tipo_Producto")) == "service")
    return df


rq_sin_oc = {nombre: calcular_rq_sin_oc(nombre) for nombre in EMPRESAS}
for nombre, df in rq_sin_oc.items():
    print(f"{nombre}: {len(df):,} líneas de RQ sin OC | S/ {df['Valor_Pendiente'].sum():,.0f}")

##### 7. Saldo disponible por SKU, centro de costo y año

In [ ]:
COLUMNAS_SALDO = [
    "Cod_Grupo_Sku", "Cod_Producto", "Producto", "Nro_Parte", "Marca", "Familia", "Cuenta Analítica",
    "Situación presupuestaria", "Periodo", "Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und",
    "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Gasto_Sin_OC", "Gasto_Rendicion", "Saldo",
    "Por_Facturacion", "Observaciones",
]


def armar_saldo(nombre):
    claves = ["Periodo", "product_id", "centro_id"]
    df_oc = calculo_oc[nombre]
    df_oc = df_oc[df_oc["confirmada"] & df_oc["Periodo"].isin(ANIOS)]

    """
        1. CADA FUENTE AGRUPADA A SKU + CENTRO + AÑO
    """
    oc = df_oc.groupby(claves, dropna=False).agg(
        Compras=("Valor_Ejecutado", "sum"), Compras_und=("Cant_Ejecutada", "sum"),
        RQ=("Valor_OC", "sum"), RQ_und=("Cant_OC", "sum"),
        RQ_COC=("Valor_Pendiente", "sum"), RQ_COC_und=("Cant_Pendiente", "sum"),
        serv_oc=("es_servicio", "any"), obs_oc=("Observaciones", unir_observaciones)).reset_index()
    soc = rq_sin_oc[nombre].groupby(claves, dropna=False).agg(
        RQ_SOC=("Valor_Pendiente", "sum"), RQ_SOC_und=("Cant_Pendiente", "sum"),
        serv_soc=("es_servicio", "any"), obs_soc=("Observaciones", unir_observaciones)).reset_index()
    gasto = gasto_sin_oc[nombre]
    gasto = gasto[gasto["Periodo"].isin(ANIOS)].groupby(claves, dropna=False).agg(
        Gasto_Sin_OC=("Gasto_Sin_OC", "sum"), Gasto_Rendicion=("Gasto_Rendicion", "sum"),
        obs_gasto=("obs", unir_observaciones)).reset_index()
    pres = presupuesto[nombre][["Periodo", "product_id", "centro_id", "Situación presupuestaria", "Presupuesto"]]

    """
        2. UNIÓN DE TODAS LAS FUENTES (CLAVES NUMÉRICAS PARA QUE EL CRUCE NO FALLE CON VACÍOS)
    """
    def normalizar(parte):
        parte = parte.copy()
        for clave in claves:
            parte[clave] = pd.to_numeric(parte[clave], errors="coerce").astype("float64")
        return parte

    df = normalizar(pres)
    for parte in [oc, soc, gasto]:
        df = df.merge(normalizar(parte), on=claves, how="outer")
    df["Periodo"] = df["Periodo"].astype(int)

    medidas = ["Presupuesto", "Compras", "Compras_und", "RQ", "RQ_und", "RQ_COC", "RQ_COC_und", "RQ_SOC", "RQ_SOC_und", "Gasto_Sin_OC", "Gasto_Rendicion"]
    df[medidas] = df[medidas].fillna(0.0)
    df["Saldo"] = df["Presupuesto"] - (df["Compras"] + df["RQ_COC"] + df["RQ_SOC"] + df["Gasto_Sin_OC"])
    df["Por_Facturacion"] = np.where(df[["serv_oc", "serv_soc"]].fillna(False).any(axis=1), "Sí", "No")

    def observaciones(r):
        partes = [r.get("obs_oc"), r.get("obs_soc"), r.get("obs_gasto")]
        if r["Presupuesto"] == 0 and (r["Compras"] + r["RQ_COC"] + r["RQ_SOC"] + r["Gasto_Sin_OC"]) != 0:
            partes.append("Sin presupuesto para este SKU y centro")
        if pd.isna(r["centro_id"]):
            partes.append("Sin centro de costo")
        return unir_observaciones(pd.Series(partes))

    df["Observaciones"] = df.apply(observaciones, axis=1)

    """
        3. DATOS DEL PRODUCTO Y DEL CENTRO
    """
    centros = {}
    for fuente in [presupuesto[nombre], gasto_sin_oc[nombre]]:
        centros.update(fuente.dropna(subset=["centro_id"]).drop_duplicates("centro_id").set_index("centro_id")["Cuenta Analítica"].to_dict())
    faltan = [c for c in df["centro_id"].dropna().unique() if c not in centros]
    if faltan:
        centros.update({c["id"]: c["display_name"] for c in leer_ids("account.analytic.account", faltan, ["display_name"], [1, 2])})
    df["Cuenta Analítica"] = df["centro_id"].map(centros)
    df = df.merge(df_productos.drop(columns=["Tipo_Producto", "udm_producto"]), on="product_id", how="left")

    df = df[COLUMNAS_SALDO].sort_values(["Periodo", "Cod_Producto", "Cuenta Analítica"]).reset_index(drop=True)
    return df


saldo = {nombre: armar_saldo(nombre) for nombre in EMPRESAS}
df_saldo_tatsa, df_saldo_tvl = saldo["TATSA"], saldo["TVL"]
df_saldo_tatsa.head()

##### 8. Resumen por centro de costo y situación presupuestaria

In [ ]:
def resumen(df):
    return (df.groupby(["Periodo", "Cuenta Analítica"], dropna=False)
              [["Presupuesto", "Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo"]].sum()
              .round(2).reset_index())


df_resumen = pd.concat([resumen(df).assign(Empresa=nombre) for nombre, df in saldo.items()], ignore_index=True)
df_resumen = df_resumen[["Empresa"] + [c for c in df_resumen.columns if c != "Empresa"]]
pd.set_option("display.float_format", "{:,.2f}".format)
df_resumen.groupby(["Empresa", "Periodo"])[["Presupuesto", "Compras", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo"]].sum()

##### 9. Informe de RQ pendientes

Una fila por línea pendiente:
- **Con OC:** la OC está abierta y falta recibir (bienes) o facturar (servicios).
- **Sin OC:** el RQ está aprobado y todavía no tiene OC confirmada por toda la cantidad.

In [ ]:
COLUMNAS_PENDIENTES = [
    "Empresa", "Periodo", "Situacion", "Cod_RQ", "Estado_RQ", "Tipo_RQ", "Cod_OC", "Estado_OC", "Fecha_OC", "Proveedor",
    "Moneda", "Cod_Producto", "Producto", "Cuenta Analítica", "Cant_RQ", "Cant_OC", "Cant_Recibida", "Cant_Facturada",
    "Cant_Pendiente", "Precio_Unit_Soles", "Valor_Pendiente", "Por_Facturacion", "Observaciones",
]


def armar_pendientes(nombre):
    """
        1. CON OC: LÍNEAS DE OC ABIERTAS CON PENDIENTE
    """
    con_oc = calculo_oc[nombre]
    con_oc = con_oc[con_oc["confirmada"] & (con_oc["Cant_Pendiente"] > 0) & con_oc["Periodo"].isin(ANIOS)].copy()
    cant_rq = requerimientos[nombre][1].set_index("rq_line_id")["Cant_RQ"]
    con_oc["Cant_RQ"] = con_oc["rq_line_id"].map(cant_rq)
    con_oc["Situacion"] = np.where(con_oc["es_servicio"], "Con OC - pendiente de facturación", "Con OC - pendiente de entrega")

    """
        2. SIN OC: LÍNEAS DE RQ APROBADAS SIN OC POR LA CANTIDAD PENDIENTE
    """
    sin_oc = rq_sin_oc[nombre].copy()
    sin_oc["Situacion"] = "Sin OC"
    for campo in ["Cod_OC", "Estado_OC", "Fecha_OC", "Proveedor", "Moneda"]:
        sin_oc[campo] = None
    sin_oc["Cant_OC"], sin_oc["Cant_Recibida"], sin_oc["Cant_Facturada"] = sin_oc["Cant_Pedida_OC"], np.nan, np.nan

    df = pd.concat([con_oc, sin_oc], ignore_index=True)
    df["Empresa"] = nombre
    df["Precio_Unit_Soles"] = df["precio_soles"]
    df["Por_Facturacion"] = np.where(df["es_servicio"], "Sí", "No")
    df = df.merge(df_productos[["product_id", "Cod_Producto", "Producto"]], on="product_id", how="left")
    mapa_centros = {}
    for fuente in [presupuesto[nombre], gasto_sin_oc[nombre]]:
        mapa_centros.update(fuente.dropna(subset=["centro_id"]).drop_duplicates("centro_id").set_index("centro_id")["Cuenta Analítica"].to_dict())
    faltan = [c for c in df["centro_id"].dropna().unique() if c not in mapa_centros]
    if faltan:
        mapa_centros.update({c["id"]: c["display_name"] for c in leer_ids("account.analytic.account", faltan, ["display_name"], [1, 2])})
    df["Cuenta Analítica"] = df["centro_id"].map(mapa_centros)
    return df[COLUMNAS_PENDIENTES].sort_values(["Periodo", "Situacion", "Cod_RQ"]).reset_index(drop=True)


df_rq_pendientes = pd.concat([armar_pendientes(nombre) for nombre in EMPRESAS], ignore_index=True)
df_rq_pendientes.groupby(["Empresa", "Periodo", "Situacion"]).agg(Lineas=("Cod_RQ", "size"), Valor_Pendiente=("Valor_Pendiente", "sum"))

##### 10. Controles

In [ ]:
"""
    1. EL PRESUPUESTO DEL SALDO CUADRA CON LO LEÍDO DE ODOO
"""
for nombre in EMPRESAS:
    leido = presupuesto[nombre].groupby("Periodo")["Presupuesto"].sum()
    en_saldo = saldo[nombre].groupby("Periodo")["Presupuesto"].sum()
    print(nombre, "presupuesto cuadra:", bool(np.allclose(leido.reindex(ANIOS, fill_value=0), en_saldo.reindex(ANIOS, fill_value=0))))


"""
    2. CASO DE REFERENCIA: G1010460 (IMPORTACIÓN CON FACTURA EMITIDA EN EL AÑO ANTERIOR)
"""
df_saldo_tatsa[df_saldo_tatsa["Cod_Producto"] == "G1010460"][
    ["Cuenta Analítica", "Periodo", "Presupuesto", "Compras", "Compras_und", "RQ_COC", "RQ_SOC", "Gasto_Sin_OC", "Saldo", "Observaciones"]]

In [ ]:
"""
    3. FILAS CON OBSERVACIONES POR TIPO
"""
tipos = {
    "Factura emitida en otro año": "Factura emitida",
    "UdM distinta / convertida": "UdM",
    "Devolución a proveedor": "Devolución",
    "Servicio (por facturación)": "Servicio",
    "Precio RQ referencial": "referencial",
    "Gasto por rendición": "rendición",
    "Sin presupuesto": "Sin presupuesto",
    "OC cerrada no completada": "no completada",
    "OC abierta con más de 12 meses": "más de 12 meses",
}
pd.DataFrame({nombre: {t: int(df["Observaciones"].str.contains(p, regex=False).sum()) for t, p in tipos.items()}
              for nombre, df in saldo.items()})

##### 11. Exportar a Excel (opcional)

Solo se ejecuta con `EXPORTAR_EXCEL = True` en Parámetros. Crea una hoja por empresa con el saldo, una hoja `RESUMEN` y una hoja `RQ_Pendientes`.

In [ ]:
def dar_formato(hoja, df):
    # Encabezado, anchos, formato de miles y filtro
    for celda in hoja[1]:
        celda.font = Font(bold=True, color="FFFFFF")
        celda.fill = PatternFill("solid", fgColor="1F4E78")
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, columna in enumerate(df.columns, start=1):
        letra = get_column_letter(i)
        hoja.column_dimensions[letra].width = min(max(len(str(columna)), 12) + 2, 60 if columna == "Observaciones" else 40)
        if pd.api.types.is_float_dtype(df[columna]):
            for celda in hoja[letra][1:]:
                celda.number_format = "#,##0.00"
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions


if EXPORTAR_EXCEL:
    with pd.ExcelWriter(ARCHIVO_EXCEL, engine="openpyxl") as escritor:
        for nombre, df in saldo.items():
            df.to_excel(escritor, sheet_name=nombre, index=False)
            dar_formato(escritor.sheets[nombre], df)
        df_resumen.to_excel(escritor, sheet_name="RESUMEN", index=False)
        dar_formato(escritor.sheets["RESUMEN"], df_resumen)
        df_rq_pendientes.to_excel(escritor, sheet_name="RQ_Pendientes", index=False)
        dar_formato(escritor.sheets["RQ_Pendientes"], df_rq_pendientes)
    print(f"Excel generado: {ARCHIVO_EXCEL}")
else:
    print("Exportación desactivada (EXPORTAR_EXCEL = False)")